# 04 — BART Two-Stage Fine-Tuning (Colab, T4)

Runs BART fine-tuning on Google Colab free tier. **Stage 1 (headline generation)** is implemented in this version; Stage 2 cells will be added in the next section of the project.

**Runtime setup:** Runtime → Change runtime type → T4 GPU (free tier).

## Stage 1 cell flow
1. GPU verification (CUDA availability, VRAM)
2. Clone project repo
3. Install dependencies
4. Mount Google Drive (checkpoints persist across sessions)
5. Set paths + choose smoke-test vs full run
6. Prepare CNN/DM training data (calls `cnn_dm_prep.main()`)
7. Import trainer + load config
8. **Train Stage 1** — `trainer.train_stage1(...)` (training + per-epoch eval on val subset)
9. Final full-val evaluation (all metrics on 1K val split)
10. Generate + print 20 sample headlines for qualitative inspection
11. Verify best checkpoint saved to Drive

**Selection metric:** BERTScore F1 (semantic, rewards paraphrase — see [Status.md](../Status.md) for the full discussion).
**Reported metrics:** rouge1, rouge2, rougeL, eval_loss, bertscore_f1.

## 1. GPU verification

In [ ]:
import torch
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f'GPU: {props.name}')
    print(f'VRAM: {props.total_memory / 1024**3:.1f} GB')
    print(f'CUDA: {torch.version.cuda}')
else:
    raise RuntimeError('No GPU detected. Switch runtime to T4 via Runtime > Change runtime type.')

## 2. Clone the project repo

In [ ]:
import os
REPO_URL = 'https://github.com/saikumar221/NewsForge.git'
WORKDIR = '/content/NewsForge'

if os.path.isdir(WORKDIR):
    print('Repo already cloned; pulling latest commits...')
    !cd $WORKDIR && git fetch origin main && git reset --hard origin/main
else:
    !git clone $REPO_URL $WORKDIR

%cd $WORKDIR
!git log -1 --oneline

## 3. Install dependencies (Colab-safe)

Colab ships `torch`, `torchvision`, `torchaudio`, `transformers`, `datasets`, etc. as a CUDA-matched stack. Some of our extras (e.g., `sentence-transformers`, `bert-score`) declare loose `torch>=...` deps — pip will silently upgrade torch to fulfill them, breaking the CUDA match with torchvision/torchaudio.

Fix: snapshot the installed versions of the critical packages into a **pip constraints file**, then install extras with `-c <constraints>`. pip must respect the pinned versions and cannot upgrade them.

In [ ]:
# Snapshot Colab's CUDA-matched ML stack into a constraints file so pip cannot upgrade it.
import importlib.metadata

COLAB_PINS = [
    'torch', 'torchvision', 'torchaudio',
    'transformers', 'tokenizers', 'datasets',
    'numpy', 'pandas', 'scipy', 'scikit-learn',
]
pins_path = '/tmp/colab_pins.txt'
with open(pins_path, 'w') as f:
    for pkg in COLAB_PINS:
        try:
            f.write(f'{pkg}=={importlib.metadata.version(pkg)}\n')
        except importlib.metadata.PackageNotFoundError:
            pass

print('Pinned versions (pip will not upgrade these):')
!cat {pins_path}
print()

# Install extras with constraints. pip respects -c and will fail cleanly rather than upgrade pinned pkgs.
!pip install -q -c {pins_path} \
  sentence-transformers \
  hdbscan \
  umap-learn \
  bert-score \
  rouge-score \
  newsapi-python \
  python-dotenv

print('Dependencies installed (Colab stack preserved).')

## 4. Mount Google Drive

Checkpoints go to Drive so they survive session disconnects. Free Colab sessions time out after ~12h of activity or disconnect on inactivity.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE_ROOT = '/content/drive/MyDrive/NewsForge'
os.makedirs(DRIVE_ROOT, exist_ok=True)
print(f'Drive root: {DRIVE_ROOT}')

## 5. Paths + run mode

Set `SMOKE_TEST = True` for a 5K/500/500 sanity run (~30 min end-to-end). Set `False` for the 50K/1K/1K real run (~2h 15min, may need 2 sessions).

In [ ]:
SMOKE_TEST = True          # Flip to False for the full 50K run after smoke test passes

# Checkpoint persistence mode.
#   False → checkpoints live in /content/checkpoints/stage1 (fast, free, lost on session disconnect).
#           Good for smoke tests that fit in one session.
#   True  → checkpoints persist to Google Drive under DRIVE_ROOT/checkpoints/stage1/
#           (~1.6 GB best model + ~4.6 GB per intermediate checkpoint). Survives disconnects.
SAVE_TO_DRIVE = False

if SAVE_TO_DRIVE:
    STAGE1_CKPT_DIR = f'{DRIVE_ROOT}/checkpoints/stage1'
else:
    STAGE1_CKPT_DIR = '/content/checkpoints/stage1'
STAGE1_DATA_DIR = 'data/processed/cnn_dailymail/stage1'   # In-repo (re-prepped each session)

os.makedirs(STAGE1_CKPT_DIR, exist_ok=True)
print(f'Checkpoint dir:    {STAGE1_CKPT_DIR}')
print(f'  persistence:     {"Google Drive" if SAVE_TO_DRIVE else "Colab /content (ephemeral)"}')
print(f'Stage 1 data dir:  {STAGE1_DATA_DIR}')
print(f'Run mode:          {"SMOKE (5K/500/500)" if SMOKE_TEST else "FULL (50K/1K/1K)"}')

## 6. Prepare CNN/DM training data

Applies sentence-split + expanded `(CNN) --` / `CITY (CNN) --` strip, builds Stage 1 (article → headline) and Stage 2 (input → summary) `DatasetDict`s. Re-prep on each session is fine — HF caches the raw CNN/DM in Colab's local disk.

In [ ]:
if SMOKE_TEST:
    !python -m src.preprocessing.cnn_dm_prep --smoke-test
else:
    !python -m src.preprocessing.cnn_dm_prep

from datasets import load_from_disk
stage1_data = load_from_disk(STAGE1_DATA_DIR)
print({k: len(v) for k, v in stage1_data.items()})
print('Columns:', stage1_data['train'].column_names)

## 7. Import trainer + load config

All training logic lives in [src/summarization/trainer.py](../src/summarization/trainer.py) so the code is reusable outside Colab. This cell loads the project config so training args inherit from `configs/config.yaml` (T4-tuned: batch_size=4, grad_accum=8, fp16, gradient_checkpointing).

In [ ]:
from src.summarization import trainer
config = trainer.load_config()

print('Model:', config['summarization']['model_name'])
print('Training config:', config['summarization']['training'])
print('Stage 1 output tokens:', config['summarization']['stage1']['max_output_tokens'])
print('Generation:', config['generation'])
print('Selection metric: bertscore_f1 (model:', config['evaluation']['bertscore_model'], ')')

## 8. Train Stage 1

Runs the full training loop. Per-epoch eval on a 500-example val subset (keeps each epoch tight). Best checkpoint selected by `bertscore_f1` and restored at end. Training logs include rouge1/rouge2/rougeL/eval_loss/bertscore_f1.

Expected wall time on T4:
- Smoke test (5K train): ~15–20 minutes
- Full run (50K train): ~2 hours

If the Colab session disconnects, re-run cells 1–7 then resume by pointing the Trainer at `STAGE1_CKPT_DIR` (HF will pick up the last checkpoint).

In [ ]:
result = trainer.train_stage1(
    config=config,
    stage1_data_dir=STAGE1_DATA_DIR,
    output_dir=STAGE1_CKPT_DIR,
    eval_subset_size=500,
)
print('Best checkpoint:', result['best_model_checkpoint'])
print('Training loss:', result['train_result'].training_loss)

## 9. Full-validation evaluation

Already computed as part of `train_stage1()` — this cell just prints the results cleanly. The reported metrics span the full 1K val split (vs the 500-subset used per epoch).

In [ ]:
final_metrics = result['final_full_val_metrics']
print('=== Stage 1 final full-val metrics ===')
for k in sorted(final_metrics):
    v = final_metrics[k]
    if isinstance(v, float):
        print(f'  {k:<35} {v:.4f}')
    else:
        print(f'  {k:<35} {v}')

## 10. Qualitative inspection — 20 sample headlines

Generates headlines for 20 validation articles using beam search (config `generation` block) so you can spot-check for the failure modes we flagged:
- vagueness / loss of specificity
- Reuters-dateline leakage (our strip rule is CNN-only)
- length clipping at `max_output_tokens: 48`
- over-extraction (copies article opening verbatim)

In [ ]:
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

best_ckpt = result['best_model_checkpoint']
print(f'Loading best checkpoint from: {best_ckpt}')
model = AutoModelForSeq2SeqLM.from_pretrained(best_ckpt)
tokenizer = AutoTokenizer.from_pretrained(best_ckpt)

val = stage1_data['validation'].shuffle(seed=42).select(range(20))
articles = val['article']
refs = val['headline']

preds = trainer.generate_headlines(
    model=model,
    tokenizer=tokenizer,
    articles=articles,
    generation_cfg=config['generation'],
    max_input_tokens=config['summarization']['stage1']['max_input_tokens'],
    max_output_tokens=config['summarization']['stage1']['max_output_tokens'],
    batch_size=4,
)

for i, (article, ref, pred) in enumerate(zip(articles, refs, preds), 1):
    print(f'--- {i}/20 ---')
    print(f'ARTICLE[:180]: {article[:180]}')
    print(f'REF:  {ref}')
    print(f'PRED: {pred}')
    print()

## 11. Verify best checkpoint is on Drive

Sanity-check that the checkpoint we just trained is persisted to Drive and not just Colab's ephemeral disk. If the session disconnects after this cell, you should be able to resume from Drive in a new session.

In [ ]:
import pathlib

ckpt_root = pathlib.Path(STAGE1_CKPT_DIR)
print(f'Checkpoint root: {ckpt_root}')
print(f'Exists: {ckpt_root.exists()}')
if ckpt_root.exists():
    total_bytes = sum(p.stat().st_size for p in ckpt_root.rglob("*") if p.is_file())
    print(f'Total size: {total_bytes / 1024**2:.1f} MB')
    print('Top-level entries:')
    for entry in sorted(ckpt_root.iterdir()):
        print(f'  {entry.name}')

print('\nTo resume from this checkpoint in a fresh session:')
print(f'  AutoModelForSeq2SeqLM.from_pretrained({STAGE1_CKPT_DIR!r})')

---
# 📝 Stage 2 — Summary Generation

Fine-tunes a **separate** `facebook/bart-large-cnn` from scratch on the Stage 2 data (`data/processed/cnn_dailymail/stage2/`) which already has `input = "{reference_headline}\n{article}"` and `target = joined highlights`.

**Key differences from Stage 1:**
- **Selection metric:** `rougeLsum` (HF's canonical summarization metric; per-sentence LCS average) instead of BERTScore F1 — conventional choice for summary evaluation
- **Generation params:** `generation_stage2` block — `length_penalty=2.0`, `min_length=30` (BART's published CNN/DM summarization defaults)
- **Output length:** up to 128 tokens (vs Stage 1's 48)
- **Input length:** same 1024-token cap

Stages are trained **independently** from the public CNN/DM checkpoint. At training time Stage 2 sees *reference* (first-highlight) headlines prepended to the article. At inference time you'd use Stage 1's generated headline instead — done in the Automatic Evaluation section later.

In [ ]:
# Stage 2 paths + persistence toggle (mirrors Stage 1's layout).
SAVE_TO_DRIVE_STAGE2 = False  # True → persistent on Drive; False → local /content (ephemeral)

if SAVE_TO_DRIVE_STAGE2:
    STAGE2_CKPT_DIR = f'{DRIVE_ROOT}/checkpoints/stage2'
else:
    STAGE2_CKPT_DIR = '/content/checkpoints/stage2'
STAGE2_DATA_DIR = 'data/processed/cnn_dailymail/stage2'

os.makedirs(STAGE2_CKPT_DIR, exist_ok=True)
print(f'Stage 2 checkpoint dir: {STAGE2_CKPT_DIR}')
print(f'  persistence:          {"Google Drive" if SAVE_TO_DRIVE_STAGE2 else "Colab /content (ephemeral)"}')
print(f'Stage 2 data dir:       {STAGE2_DATA_DIR}')
print(f'Generation (Stage 2):   {config["generation_stage2"]}')
print(f'Selection metric:       rougeLsum')

## Stage 2 — Train

Runs the full training loop. Per-epoch eval on a 500-example val subset using `rougeLsum` for best-model selection; `rouge1/rouge2/rougeL/rougeLsum/bertscore_f1` all logged.

Expected wall time on T4 (smoke test): ~15–20 minutes. Full 50K run: ~2 hours.

In [ ]:
import importlib
from src.summarization import trainer as _trainer
importlib.reload(_trainer)  # Pick up any new functions (e.g. train_stage2) after a git pull

result_stage2 = _trainer.train_stage2(
    config=config,
    stage2_data_dir=STAGE2_DATA_DIR,
    output_dir=STAGE2_CKPT_DIR,
    eval_subset_size=500,
)
print('Best checkpoint:', result_stage2['best_model_checkpoint'])
print('Training loss:', result_stage2['train_result'].training_loss)

## Stage 2 — Full-validation metrics + qualitative inspection

In [ ]:
final_metrics_s2 = result_stage2['final_full_val_metrics']
print('=== Stage 2 final full-val metrics ===')
for k in sorted(final_metrics_s2):
    v = final_metrics_s2[k]
    if isinstance(v, float):
        print(f'  {k:<35} {v:.4f}')
    else:
        print(f'  {k:<35} {v}')

In [ ]:
from datasets import load_from_disk
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

stage2_data = load_from_disk(STAGE2_DATA_DIR)
best_ckpt_s2 = result_stage2['best_model_checkpoint']
print(f'Loading Stage 2 best checkpoint from: {best_ckpt_s2}')
model_s2 = AutoModelForSeq2SeqLM.from_pretrained(best_ckpt_s2)
tokenizer_s2 = AutoTokenizer.from_pretrained(best_ckpt_s2)

val = stage2_data['validation'].shuffle(seed=42).select(range(20))
preds_s2 = _trainer.generate_summaries(
    model=model_s2,
    tokenizer=tokenizer_s2,
    inputs=val['input'],
    generation_cfg=config['generation_stage2'],
    max_input_tokens=config['summarization']['stage2']['max_input_tokens'],
    max_output_tokens=config['summarization']['stage2']['max_output_tokens'],
    batch_size=2,
)

for i, (headline, target, pred) in enumerate(zip(val['headline'], val['target'], preds_s2), 1):
    print(f'--- {i}/20 ---')
    print(f'REF HEADLINE: {headline}')
    print(f'REF SUMMARY:  {target[:260]}{"..." if len(target) > 260 else ""}')
    print(f'PRED SUMMARY: {pred}')
    print()

In [ ]:
import pathlib

s2_root = pathlib.Path(STAGE2_CKPT_DIR)
print(f'Stage 2 checkpoint root: {s2_root}')
print(f'Exists: {s2_root.exists()}')
if s2_root.exists():
    total_bytes = sum(p.stat().st_size for p in s2_root.rglob('*') if p.is_file())
    print(f'Total size: {total_bytes / 1024**2:.1f} MB')
    print('Top-level entries:')
    for entry in sorted(s2_root.iterdir()):
        print(f'  {entry.name}')

print('\nTo resume in a fresh session:')
print(f'  AutoModelForSeq2SeqLM.from_pretrained({STAGE2_CKPT_DIR!r})')